# 02 — Text Analytics Pipeline: F0 → F5

**Clay Harris (jbm2rt@virginia.edu) / Text as Data / 2026-05-07**

This notebook runs the full text analytics pipeline on the papal encyclicals corpus.
Each stage transforms the data and adds annotations, culminating in three unsupervised models.

| Stage | Input → Output | Key Tables |
|-------|----------------|------------|
| F1 | Raw text → paragraphs | F1 corpus |
| F2 | Paragraphs → tokens | LIBRARY, TOKEN, VOCAB |
| F3 | Tokens → annotations | TOKEN+, VOCAB+, DOC_SENTIMENT |
| F4 | Annotations → weights | TFIDF_DTM |
| F5 | Weights → models | DOC_PCA, DOC_TOPICS, EMBEDDINGS |

In [1]:
# ── Configuration ──────────────────────────────────────────────────────
ENGLISH_ONLY  = True
N_COMPONENTS  = 10
N_TOPICS      = 10
W2V_DIM       = 100
FORCE_RERUN   = False   # Set True to recompute all stages from scratch

In [2]:
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))

from src.pipeline import (
    build_f1_corpus, build_f2_tables, build_f3_annotations,
    build_f4_tfidf, build_f5_models, save_tables,
    cache_exists, save_cache, load_cache,
    PROCESSED_DIR, CACHE_DIR,
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print('Imports OK')

Imports OK


## F1: Machine Learning Corpus Format

Reads raw `.txt` files from `data/raw/` and the metadata index `encyclicals_index.json`.
Each document is split into paragraphs (the minimum discursive unit), producing one row per
paragraph with columns `doc_id`, `pope`, `title`, `year`, `para_num`, and `para_text`.
Setting `ENGLISH_ONLY=True` restricts processing to documents detected as English by the scraper.

In [3]:
%%time
if not FORCE_RERUN and cache_exists('f1'):
    corpus = load_cache('f1')
else:
    corpus = build_f1_corpus(english_only=ENGLISH_ONLY)
    save_cache('f1', corpus)

2026-05-07 09:52:16,418 [INFO] Building F1 corpus from raw text files...
2026-05-07 10:00:22,554 [INFO] F1 corpus: 407753 paragraphs from 17384 documents
2026-05-07 10:00:22,990 [INFO] Checkpoint saved: f1


CPU times: total: 15.1 s
Wall time: 8min 6s


In [4]:
print(f'CORPUS docs:  {corpus["doc_id"].nunique():,}')
print(f'CORPUS paras: {len(corpus):,}')
corpus.head()

CORPUS docs:  17,384
CORPUS paras: 407,753


,doc_id,pope,title,year,para_num,para_text
0,church_councils__the_fifth_general_council_of_...,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,0,INTRODUCTION
1,church_councils__the_fifth_general_council_of_...,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,1,This council was summoned by pope Julius II by...
2,church_councils__the_fifth_general_council_of_...,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,2,There were twelve sessions. The first five of ...
3,church_councils__the_fifth_general_council_of_...,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,3,"All the decrees of this council, at which the ..."
4,church_councils__the_fifth_general_council_of_...,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,4,The decisions on the reform of the curia produ...


## F2: STADM Tables — LIBRARY, TOKEN, VOCAB

Applies NLTK sentence and word tokenization to each paragraph, producing the three core
Standard Text Analytic Data Model tables:

- **LIBRARY** — one row per document with metadata and aggregate statistics
- **TOKEN** — one row per token, indexed by the OHCO hierarchy `(doc_id, para_num, sent_num, token_num)`
- **VOCAB** — one row per unique term, with raw frequency and document frequency

In [5]:
%%time
if not FORCE_RERUN and cache_exists('f2'):
    LIBRARY, TOKEN, VOCAB = load_cache('f2')
else:
    LIBRARY, TOKEN, VOCAB = build_f2_tables(corpus)
    save_cache('f2', (LIBRARY, TOKEN, VOCAB))

2026-05-07 10:00:23,108 [INFO] Building F2 STADM tables (LIBRARY, TOKEN, VOCAB)...
Tokenizing: 100%|██████████| 407753/407753 [06:33<00:00, 1035.50it/s]
2026-05-07 10:08:19,037 [INFO] LIBRARY: 17384 documents
2026-05-07 10:08:19,038 [INFO] TOKEN:   28589033 tokens
2026-05-07 10:08:19,039 [INFO] VOCAB:   225720 unique terms
2026-05-07 10:08:44,703 [INFO] Checkpoint saved: f2


CPU times: total: 7min 24s
Wall time: 8min 21s


In [6]:
print(f'LIBRARY: {len(LIBRARY):,} documents')
print(f'TOKEN:   {len(TOKEN):,} tokens')
print(f'VOCAB:   {len(VOCAB):,} unique terms')
LIBRARY.head()

LIBRARY: 17,384 documents
TOKEN:   28,589,033 tokens
VOCAB:   225,720 unique terms


,pope,title,year,n_paragraphs,n_chars,n_tokens
doc_id,,,,,,
church_councils__the_fifth_general_council_of_the_lateran__1512_17,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,196,211930,40440
church_councils__the_first_general_council_of_constantinople__381,Church Councils,"The First General Council of Constantinople, 381",,51,22891,4427
church_councils__the_first_general_council_of_lyons__1245,Church Councils,"The First General Council of Lyons, 1245",1245,94,77512,15108
church_councils__the_first_general_council_of_nicaea__325,Church Councils,"The First General Council of Nicaea, 325",,20,6097,1169
church_councils__the_first_general_council_of_the_lateran__1123,Church Councils,"The First General Council of the Lateran, 1123",,37,16643,3527


In [7]:
TOKEN.head(10)

,doc_id,para_num,sent_num,token_num,token_str,term_str
token_id,,,,,,
0,church_councils__the_fifth_general_council_of_...,0,0,0,INTRODUCTION,introduction
1,church_councils__the_fifth_general_council_of_...,1,0,0,This,this
2,church_councils__the_fifth_general_council_of_...,1,0,1,council,council
3,church_councils__the_fifth_general_council_of_...,1,0,2,was,was
4,church_councils__the_fifth_general_council_of_...,1,0,3,summoned,summoned
5,church_councils__the_fifth_general_council_of_...,1,0,4,by,by
6,church_councils__the_fifth_general_council_of_...,1,0,5,pope,pope
7,church_councils__the_fifth_general_council_of_...,1,0,6,Julius,julius
8,church_councils__the_fifth_general_council_of_...,1,0,7,II,ii


In [8]:
VOCAB.head(20)

,n,df,idf
term_str,,,
the,1907282,17306,0.004497
",",1622820,17373,0.000633
of,1246661,17305,0.004555
.,947770,17220,0.009479
and,898167,17261,0.007101
to,842630,17284,0.005769
in,590497,17325,0.003400
a,343137,17137,0.014310
is,341874,16638,0.043861


## F3: NLP Annotations — POS, Lemma, Stopwords, Sentiment

Enriches TOKEN and VOCAB with linguistic annotations:

- **POS tags** via NLTK averaged perceptron tagger (Penn Treebank tagset)
- **Lemmas** via NLTK WordNetLemmatizer (using POS context to disambiguate)
- **Stopword flags** from NLTK English stopword list
- **VADER sentiment** on each vocabulary term (neg/neu/pos/compound)
- **DOC_SENTIMENT** — aggregate VADER compound/pos/neg/neu scores per document

This is the most compute-intensive stage. The pickle checkpoint makes reruns near-instant.

In [ ]:
%%time
if not FORCE_RERUN and cache_exists('f3'):
    LIBRARY, TOKEN, VOCAB, DOC_SENTIMENT = load_cache('f3')
else:
    LIBRARY, TOKEN, VOCAB, DOC_SENTIMENT = build_f3_annotations(TOKEN, VOCAB, LIBRARY)
    save_cache('f3', (LIBRARY, TOKEN, VOCAB, DOC_SENTIMENT))

2026-05-07 10:08:44,762 [INFO] Building F3 NLP annotations...
2026-05-07 10:08:45,128 [INFO]   POS tagging...
POS tagging: 100%|██████████| 1172783/1172783 [24:57<00:00, 783.05it/s] 
2026-05-07 10:34:35,908 [INFO]   Lemmatizing...
2026-05-07 11:27:16,329 [INFO]   Updating VOCAB...
2026-05-07 11:30:13,842 [INFO]   Computing VADER sentiment for vocab...
2026-05-07 11:30:27,866 [INFO]   Computing document-level sentiment...


In [ ]:
print(f'TOKEN cols:    {list(TOKEN.columns)}')
print(f'VOCAB cols:    {list(VOCAB.columns)}')
print(f'DOC_SENTIMENT: {len(DOC_SENTIMENT):,} documents')
TOKEN.head(10)

In [ ]:
VOCAB.head(10)

In [ ]:
print('DOC_SENTIMENT (document-level VADER scores):')
DOC_SENTIMENT.head(10)

## F4: TFIDF Vectorization

Computes normalized term frequency (TF), inverse document frequency (IDF), and TFIDF for
every (document, term) pair. Produces the **TFIDF_DTM** document-term matrix, which is the
primary input to the unsupervised models in F5.

Only meaningful terms (alphabetic, non-stop, `df > 1`) are included as DTM columns
to control dimensionality.

In [ ]:
%%time
if not FORCE_RERUN and cache_exists('f4'):
    LIBRARY, TOKEN, VOCAB, TFIDF_DTM = load_cache('f4')
else:
    LIBRARY, TOKEN, VOCAB, TFIDF_DTM = build_f4_tfidf(TOKEN, VOCAB, LIBRARY)
    save_cache('f4', (LIBRARY, TOKEN, VOCAB, TFIDF_DTM))

In [ ]:
print(f'TFIDF_DTM shape: {TFIDF_DTM.shape}  (docs x terms)')
print(f'\nTop TFIDF terms — first 5 documents:')
for doc_id in TFIDF_DTM.index[:5]:
    top = TFIDF_DTM.loc[doc_id].nlargest(5)
    print(f'  {doc_id[:50]}: {", ".join(top.index)}')
TFIDF_DTM.head()

## F5: Unsupervised Models — PCA, LDA, Word2Vec

Fits three complementary unsupervised models:

1. **PCA** — principal component analysis on the TFIDF matrix. Reveals the primary axes of
   topical variation. Outputs `DOC_PCA`, `LOADINGS`, and `explained_variance`.

2. **LDA** — Latent Dirichlet Allocation topic model. Finds soft clusters of co-occurring terms
   and assigns each document a mixture of topics. Outputs `DOC_TOPICS` and `TOPIC_TERMS`.

3. **Word2Vec** — neural term embeddings trained on lemmatized sentences. Captures semantic
   similarity in a dense vector space. Outputs `EMBEDDINGS`.

In [ ]:
%%time
if not FORCE_RERUN and cache_exists('f5'):
    f5_results = load_cache('f5')
else:
    f5_results = build_f5_models(
        LIBRARY, TOKEN, VOCAB, TFIDF_DTM,
        n_components=N_COMPONENTS,
        n_topics=N_TOPICS,
        w2v_dim=W2V_DIM,
    )
    save_cache('f5', f5_results)

In [ ]:
DOC_PCA = f5_results['DOC_PCA']
LOADINGS = f5_results['LOADINGS']
explained_variance = f5_results['explained_variance']

print(f'DOC_PCA shape: {DOC_PCA.shape}')
print(f'Explained variance (cumulative): {explained_variance.cumsum().round(3).tolist()}')
DOC_PCA.head()

In [ ]:
DOC_TOPICS = f5_results['DOC_TOPICS']
TOPIC_TERMS = f5_results['TOPIC_TERMS']

print(f'DOC_TOPICS shape: {DOC_TOPICS.shape}')
print('\nTop terms per topic:')
for col in TOPIC_TERMS.columns:
    top = TOPIC_TERMS[col].nlargest(10).index.tolist()
    print(f'  {col}: {", ".join(top)}')
DOC_TOPICS.head()

In [ ]:
EMBEDDINGS = f5_results['EMBEDDINGS']
print(f'EMBEDDINGS shape: {EMBEDDINGS.shape}  (terms x dimensions)')
EMBEDDINGS.head()

## Save All Deliverables

Write all tables to `data/processed/*.csv` and print a manifest of output files.

In [ ]:
save_tables(LIBRARY, TOKEN, VOCAB, TFIDF_DTM, DOC_SENTIMENT=DOC_SENTIMENT,
            f5_results=f5_results)

print('\n── Output manifest ───────────────────────────────────────────────')
for f in sorted(PROCESSED_DIR.glob('*.csv')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<35} {size_kb:>8.0f} KB')
print('────────────────────────────────────────────────────────────────')
print('Pipeline complete!')